# S. aureus / NR4A — ZWO camera

**Makes up Figure(s):** TBD — fill in once the paper's final figure numbering is set
(see `claude/paper_figures_reproduction.md` §4 question 1).

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/ZWO/20260624_SAureus_NR4A/FOV3/`
— *S. aureus* labelled with NR4A, imaged on the ZWO (ASI585MC) camera.

**Source:** combines two `developer`-branch dev notebooks (`notebooks/saureus/
SAureus_Analysis_ZWO_RawAnalysis.ipynb` + `SAureus_ZWO_PostAnalysis.ipynb`) into a single
download → raw fit → post-analysis pipeline, runnable by an external reader with no
access to the author's own machine. Neither dev notebook is a straight copy/paste — see
"What changed while porting" below.

## FAST_MODE

Each raw acquisition file is a ~4.3 GB multi-frame OME-TIF; the full FOV3 acquisition is
27 such files (~230 GB). `FAST_MODE = True` (default) downloads and fits only the first
file (~4.3 GB, ~260 frames) — enough to run the whole pipeline end-to-end and see
sensible-looking output, but with far less signal than the published numbers. Set
`FAST_MODE = False` to download and process the complete FOV3 acquisition (**~230 GB,
hours of download + fitting** — run in `tmux`/`screen`).

## What changed while porting (not just API updates)

- **Two real bugs found and fixed, not carried over:**
  1. The post-analysis dev notebook hardcodes `69` (nm) as the pixel size everywhere —
     that's the **Ximea** camera's pixel size. This is the **ZWO** camera (0.0715 µm =
     **71.5 nm**, see `src/CameraDefaults.py`), and its own raw-fit dev notebook correctly
     uses `camera="zwo"` throughout. The post-analysis notebook also loads calibration
     maps from `Camera_Calibrations/Ximea_Camera/` (dead code — see next point — but the
     wrong-camera intent is the same bug). Fixed here: a single `PIXEL_SIZE_NM = 71.5`
     constant used everywhere downstream (drift, linking, rendering, FRC).
  2. The Nile Red wavelength-fitting step (`fit_wavelengths_pixelated`) builds its own
     `camera_params['pixel_QYs']` from a *default* (Ximea) `SpectralFunctions.Spectral_Funcs()`
     instead of the ZWO-configured one — meaning the wavelength fit would have been done
     against the wrong sensor's QE curves. Fixed here: `Spectral_Funcs(camera="zwo")`.
- **Dead code dropped:** the post-analysis notebook loads `gain_map`/`offset_map`/
  `variance`/`read_noise`/`rqe`/`camera_parameters` and instantiates `SRes_F`/`M_F`/`SD_F`
  that are never referenced again (post-analysis works entirely from the already-fit `.h5`
  localisation table) — confirmed by grepping every cell before dropping them, not just
  assumed.
- **No `metadata.txt` sidecar:** the EBI upload notebooks only hashed `.tif`/`.tiff` files
  (`claude/LOG.md`, EBI Update session), so the ImageJ-style `metadata.txt` the dev
  notebooks read `width`/`height`/`n_frames` from isn't part of the deposition. Both are
  derived here directly from the downloaded data instead: `width`/`height` from the first
  frame's own shape (`IO.read_tiff`), `n_frames` from the fitted localisations' own
  `frame` column max.
- **Output paths**: personal `/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks`
  SVG saves replaced with a local `figures/` folder beside this notebook (`*.svg` already
  gitignored); downloaded raw data + fit outputs go to a local `data/` folder
  (`notebooks/figures/data/` is gitignored — never commit downloaded raw data).
- **FOV choice:** the two dev notebooks disagree — raw-analysis used `FOV2`, post-analysis
  used `FOV3` (and is the one that saved real `For_Talks` SVGs). This notebook uses `FOV3`
  throughout, on the assumption that's the one that actually produced real output; flag if
  that's wrong.

## Verification

I have not run this against the real full-scale download (`FAST_MODE = False`) — the
pure-Python porting (imports, API signatures, dead-code removal) is checked against the
current `src/` API, but the actual published numbers (FIRE resolution, wavelength
precision, per-region spectral separation) need the user's own check once run for real.
The two author-tuned pixel regions in §10 (`rects`) and the crop windows in §6/§9 are
copied verbatim from the original notebook and will very likely need re-tuning against
FOV3's real rendered image.


---
## 0 — Imports


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colors as mcolors
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd()))
import _biostudies_download as bd

from pyS3M import IOFunctions
from pyS3M import HelperFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SpectralFunctions
from pyS3M import MaskFunctions
from pyS3M import SpotDetectionFunctions
from pyS3M import SR_Functions
from pyS3M import PlottingBase
from pyS3M import render
from pyS3M import postprocess as _postprocess
from pyS3M import NileRedFunctions
from pyS3M.DriftCorrectionFunctions import Drift_Correction_Functions
from pyS3M.LinkingFunctions import link_localisations
from pyS3M.FRCFunctions import fire

IO = IOFunctions.IO_Functions()
H_F = HelperFunctions.Helper_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)


---
## 1 — Configuration


In [ ]:
FAST_MODE = True   # True: 1 file (~4.3 GB, illustrative). False: full FOV3 (~230 GB, real numbers).

# ── ZWO camera: pixel size (nm) and spectral/mask/spot-detection setup ───────────────────
PIXEL_SIZE_NM = 71.5   # ZWO (ASI585MC): 0.0715 um -- see src/CameraDefaults.py.
                        # (The dev post-analysis notebook hardcoded 69 nm, the Ximea value --
                        # see "What changed while porting" above.)

S_F  = SpectralFunctions.Spectral_Funcs(camera="zwo")
M_F  = MaskFunctions.Mask_Functions(camera="zwo")
SD_F = SpotDetectionFunctions.SpotDetection_Functions(camera="zwo")

# camera="zwo" alone only sets SuperRes_Functions' own pixel_size/mosaic_unit -- its
# internal default mask_functions/spot_detection_functions sub-instances would otherwise
# silently stay ximea-configured (they don't inherit the camera argument), so inject the
# already-zwo-configured M_F/SD_F explicitly (same pattern the raw-analysis dev notebook
# used).
SupRes_F = SR_Functions.SuperRes_Functions(camera="zwo", mask_functions=M_F, spot_detection_functions=SD_F)

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])
camera_params = {"pixel_QYs": pixel_QYs, "wavelength": wavelength}

import types
smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

# ── Paths ──────────────────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd().parents[1] if (Path.cwd().parents[1] / "Camera_Calibrations").exists() else Path.cwd()
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "ZWO_Camera"
DATA_DIR  = Path("data") / "SAureus_ZWO_FOV3"
FIG_DIR   = Path("figures")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

gain_map    = IO.read_tiff(CALIB_DIR / "gain.tif")
offset_map  = IO.read_tiff(CALIB_DIR / "offset.tif")
variance    = IO.read_tiff(CALIB_DIR / "variance.tif")
read_noise  = IO.read_tiff(CALIB_DIR / "readnoise.tif")
rqe         = IO.read_tiff(CALIB_DIR / "rqe.tif")

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Calibration: {CALIB_DIR}")
print(f"Data dir   : {DATA_DIR.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")


---
## 2 — Download the raw data from BioStudies (`S-BIAD3210`)

Lists the real FOV3/Slide3_1 filenames on the deposition and downloads either just the
first file (`FAST_MODE`) or the complete acquisition. Re-running this cell is safe — files
already downloaded at the correct size are skipped (`_biostudies_download.download_file`).


In [ ]:
REMOTE_FOLDER = (
    "2026_Multicolour_Paper/ZWO/20260624_SAureus_NR4A/"
    "FOV3/500pM_NR4A_60perc_515_515LP650200BP_Slide3_1"
)
STEM = "500pM_NR4A_60perc_515_515LP650200BP_Slide3_1_MMStack_Pos0"

if FAST_MODE:
    remote_files = [f"{REMOTE_FOLDER}/{STEM}.ome.tif"]
else:
    # The full acquisition is split across 27 files: the base file plus _1 .. _26.
    remote_files = [f"{REMOTE_FOLDER}/{STEM}.ome.tif"] + [
        f"{REMOTE_FOLDER}/{STEM}_{i}.ome.tif" for i in range(1, 27)
    ]

print(f"Downloading {len(remote_files)} file(s) to {DATA_DIR} ...")
downloaded = bd.download_files(remote_files, DATA_DIR)
image_folder = DATA_DIR


---
## 3 — Raw fit: spot detection + fitting across the downloaded frames

Same call as the raw-analysis dev notebook (`SuperRes_Functions.fit_imaging_data`,
already current-API-compatible — no signature changes needed there). Writes
`Localisations.h5` into `image_folder`.

This can take a while even in `FAST_MODE` (one ~4.3 GB file); several hours in full mode.


In [ ]:
pfa = 1e-3
sigma = 1.5
fraction_true = 0.2
peak_wavelength = 0.65

# Quick single-frame sanity check before committing to the full fit.
fig, axs = SupRes_F.example_spots_singleframe(
    image_folder, pfa=pfa, sigma=sigma, fraction_true=fraction_true, frame_index=10,
    variance=variance, read_noise=read_noise, rqe=rqe, offset_map=offset_map,
    gain_map=gain_map, peak_wavelength=peak_wavelength,
)
plt.show()


In [ ]:
SupRes_F.fit_imaging_data(
    image_folder,
    smoothing_function,
    gain_map,
    offset_map,
    rqe,
    read_noise,
    variance=variance,
    pfa=pfa,
    ROI_size=12,
    peak_wavelength=peak_wavelength,
    fraction_true=fraction_true,
    sigma=sigma,
    NA=1.49,
    # pixel_size intentionally omitted: defaults to SupRes_F.pixel_size (zwo -> 71.5 nm).
    image_type=".tif",
)


---
## 4 — Load fit results and apply quality-control filters

`width`/`height` are read directly from the first downloaded frame (`IO.read_tiff`) since
the deposition doesn't include the ImageJ `metadata.txt` sidecar the dev notebook used.


In [ ]:
localisation_files = H_F.file_search(image_folder, ".h5", "")
print(localisation_files)

first_frame = IO.read_tiff(downloaded[0], frame=0)
height, width = first_frame.shape
print(f"Frame shape: {width} x {height} px")

loc_data = pd.read_hdf(localisation_files[0])
print(f"Raw localisations: {len(loc_data):,}")

plt.hist(loc_data["chi_sqr"], 100)
plt.xlim([0, 3])
plt.show()


In [ ]:
# Author's original QC cuts for this dataset -- copied verbatim from the dev notebook.
loc_data = loc_data[loc_data["xc_err"] < 50 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["yc_err"] < 50 / PIXEL_SIZE_NM]

loc_data = loc_data[loc_data["xc_err"] > 0]
loc_data = loc_data[loc_data["yc_err"] > 0]

loc_data = loc_data[loc_data["s_x_err"] < 50 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["s_y_err"] < 50 / PIXEL_SIZE_NM]

loc_data = loc_data[(loc_data["s_x"] > 80 / PIXEL_SIZE_NM) & (loc_data["s_x"] < 175 / PIXEL_SIZE_NM)]
loc_data = loc_data[(loc_data["s_y"] > 80 / PIXEL_SIZE_NM) & (loc_data["s_y"] < 175 / PIXEL_SIZE_NM)]

loc_data = loc_data[loc_data["A_B_err"] < 0.01]
loc_data = loc_data[loc_data["A_G_err"] < 0.01]
loc_data = loc_data[loc_data["A_R_err"] < 0.01]

loc_data = loc_data[loc_data["A_B"] > 0.005]
loc_data = loc_data[loc_data["A_G"] > 0.05]
loc_data = loc_data[loc_data["A_R"] > 0.4]

loc_data = loc_data[loc_data["bg_B_err"] < 0.15]
loc_data = loc_data[loc_data["bg_G_err"] < 0.15]
loc_data = loc_data[loc_data["bg_R_err"] < 0.15]

loc_data = loc_data[~((loc_data["photons"] < 500) & (loc_data["spot_matched_filter_response"] < 50))]
loc_data = loc_data[loc_data["spot_background_std"] > 10]
loc_data = loc_data[loc_data["chi_sqr"] < 2]

print(f"After QC filtering: {len(loc_data):,} localisations")

plt.hist(loc_data["A_R"], 1000, color="red")
plt.hist(loc_data["A_G"], 1000, alpha=0.5, color="darkgreen")
plt.xlim([0, 1])
plt.show()


---
## 5 — Drift correction (AIM) and frame-to-frame linking


In [ ]:
drift_corrector = Drift_Correction_Functions()

info = [{
    "Width": width,
    "Height": height,
    "Frames": int(loc_data["frame"].max()),
    "Pixelsize": PIXEL_SIZE_NM,
}]

corrected_locs, drift_result = drift_corrector.undrift(
    locs=loc_data.to_records(index=False),
    info=info,
    method="aim",
    segmentation=50,
    intersect_d=20 / PIXEL_SIZE_NM,
    roi_r=60 / PIXEL_SIZE_NM,
)
corrected_locs = pd.DataFrame(corrected_locs)

plt.plot(drift_result.drift_x * PIXEL_SIZE_NM)
plt.plot(drift_result.drift_y * PIXEL_SIZE_NM)
plt.show()


In [ ]:
# n_frames from the fitted data itself, not a metadata.txt sidecar (not part of the
# deposition -- see "What changed while porting" above).
n_frames = int(loc_data["frame"].max()) + 1

link_r = 0.5 * (np.median(corrected_locs["xc_err"]) + np.median(corrected_locs["yc_err"]))
linked = link_localisations(corrected_locs, n_frames=n_frames, r_max=link_r, max_dark_time=2)
print(f"Linked localisations: {len(linked):,}")

IO.write_h5_database(df=linked, filepath=localisation_files[0].split(".h5")[0] + "_Undrifted_Linked.h5",
                      normalise_photons=False)


---
## 6 — Render the linked super-resolution image


In [ ]:
oversampling = 8
min_blur_width = 4 * (PIXEL_SIZE_NM / 1000)

info = [{
    "Width": width,
    "Height": height,
    "Frames": int(linked["frame"].max()),
    "Pixelsize": PIXEL_SIZE_NM,
}]

n_locs, image_FOV = render.render(
    locs=linked.to_records(index=False),
    info=info,
    blur_method="smooth",
    oversampling=oversampling,
    min_blur_width=min_blur_width,
)[:2]

fig, axs = plotter.two_column_plot(height=2, width=2.25)
axs = plotter.image_plot(axs, image_FOV, cmap="hot", vmax=np.percentile(image_FOV, 99.9), colorbar=False)

plt.savefig(FIG_DIR / "Image_SAureus_ZWO.svg", dpi=600, format="svg")
plt.show()


---
## 7 — Segment individual bacteria from the rendered image

`postprocess.segment_locs_by_rendered_image` — image-thresholding-based aggregate
detection, much cheaper than DBSCAN for this many localisations.


In [ ]:
MIN_AREA_NM2 = 100.0
MIN_LOCS = 20

file = localisation_files[0].split(".h5")[0] + "_Undrifted_Linked.h5"

bacteria_locs, per_bacteria_stats = _postprocess.segment_locs_by_rendered_image(
    linked,
    width=width,
    height=height,
    oversampling=oversampling,
    pixel_size_nm=PIXEL_SIZE_NM,
    min_area_nm2=MIN_AREA_NM2,
    min_localisations=MIN_LOCS,
    threshold_method="li",
    callback="console",
    blur_method="gaussian",
    verbose=True,
)

bacteria_locs = bacteria_locs.rename(columns={"aggregate_id": "cluster_id", "aggregate_area_nm2": "cluster_area_nm2"})
per_bacteria_stats = per_bacteria_stats.rename(columns={"aggregate_id": "cluster_id"})

IO.write_h5_database(df=bacteria_locs, filepath=file.split(".h5")[0] + "_clusteredlocs.h5",
                      append=False, normalise_photons=False)
IO.write_h5_database(df=per_bacteria_stats, filepath=file.split(".h5")[0] + "_averagedoverclusters.h5",
                      append=False, normalise_photons=False)

print(f"Detected {bacteria_locs['cluster_id'].nunique()} bacteria")


---
## 8 — Nile Red pixelated wavelength fit


In [ ]:
nrf = NileRedFunctions.NileRed_Functions()

filters = [
    "semrock-ff01-650-200",
    "semrock-di03-r514-t1-25x36",
    "semrock-ff01-515-lp",
]

analysis_files = H_F.file_search(image_folder, "_clusteredlocs.h5", "")
MIN_LOCS_ANALYSIS = 10

for f in analysis_files:
    df_with_wavelengths, grid_info = nrf.fit_wavelengths_pixelated(
        h5_path=f,
        filter_names=filters,
        camera_parameters=camera_params,   # zwo-configured (see "What changed" above)
        output_path=f,
        min_localisations=MIN_LOCS_ANALYSIS,
        pixel_size_nm=50,
        aggregate_id_column="cluster_id",
    )

df_with_wavelengths = pd.read_hdf(analysis_files[0])
print(f"Wavelength-fitted rows: {len(df_with_wavelengths):,}")


---
## 9 — Colour-by-wavelength render


In [ ]:
n_locs, image_FOV_grey, image_FOV_colour = render.render(
    locs=df_with_wavelengths.to_records(index=False),
    oversampling=oversampling,
    viewport=((0, 0), (np.max(df_with_wavelengths["yc"]), np.max(df_with_wavelengths["xc"]))),
    blur_method="gaussian_colour",
    cparam="wl_pixel",
    c_min=np.percentile(df_with_wavelengths["wl_pixel"], 5),
    c_max=np.percentile(df_with_wavelengths["wl_pixel"], 95),
)

vmax = np.percentile(image_FOV_grey, 99.5)
vmin = np.percentile(image_FOV_grey, 1)
brightness = np.clip((image_FOV_grey - vmin) / (vmax - vmin), 0, 1)

hsv = mcolors.rgb_to_hsv(image_FOV_colour)
hsv[..., 2] = brightness
image_adjusted = mcolors.hsv_to_rgb(hsv)

fig, axs = plotter.two_column_plot(height=2, width=2.25)
axs = plotter.colour_image_plot(
    axs, image_adjusted, cbar="on", cbarlabel=r"mean wavelength/nm", cmap="jet",
    c_min=np.percentile(df_with_wavelengths["wl_pixel"], 5),
    c_max=np.percentile(df_with_wavelengths["wl_pixel"], 95),
    pixelsize=PIXEL_SIZE_NM / oversampling,
)
plt.savefig(FIG_DIR / "Image_SAureus_Wavelength.svg", dpi=600, format="svg")
plt.show()


---
## 10 — Two-region spectral comparison

`rects` are the original author's hand-picked pixel regions for FOV3, copied verbatim —
**these will very likely need re-picking against the real rendered image** once run for
real (flagged in the notebook intro's Verification note).


In [ ]:
def get_region_localisations(df, rectangles):
    """Filter localisations within each rectangle (camera pixels)."""
    regions = []
    for rect in rectangles:
        mask = (
            (df["xc"] >= rect["x0"]) & (df["xc"] < rect["x0"] + rect["w"]) &
            (df["yc"] >= rect["y0"]) & (df["yc"] < rect["y0"] + rect["h"])
        )
        regions.append(df[mask].copy())
    return regions


# Author-selected regions (absolute camera pixels) -- see markdown note above.
rects = [
    dict(x0=351, y0=425, w=4, h=5, color="#4363d8", label="Region A"),
    dict(x0=355, y0=424, w=4, h=5, color="#e6194b", label="Region B"),
]

region_A, region_B = get_region_localisations(df_with_wavelengths, rects)
print(f"Region A: {len(region_A)} localisations")
print(f"Region B: {len(region_B)} localisations")


In [ ]:
fig, axs = plotter.one_column_plot(width=2, height=1.5)

axs = plotter.histogram_plot(axs, region_A["A_R"], np.histogram_bin_edges(region_A["A_R"], "fd"),
                              density=False, color="#4363d8", alpha=0.5)
axs = plotter.histogram_plot(axs, region_B["A_R"], np.histogram_bin_edges(region_B["A_R"], "fd"),
                              density=False, color="#e6194b", alpha=0.5, xlabel="pixel 3 QE")

axs.vlines(ymin=0, ymax=100, x=np.mean(region_A["A_R"]), color="#4363d8", ls="--", lw=1)
axs.vlines(ymin=0, ymax=100, x=np.mean(region_B["A_R"]), color="#e6194b", ls="--", lw=1)

axs.set_ylim([0, 95])
plt.xlim([0.5, 0.9])
plt.savefig(FIG_DIR / "SAureus_Histogram.svg", dpi=600, format="svg")
plt.show()


---
## 11 — Per-pixel wavelength distribution


In [ ]:
averaged = df_with_wavelengths.groupby("pixel_ix", as_index=False).agg(
    xc=("xc", "mean"),
    yc=("yc", "mean"),
    A_R=("A_R", "mean"),
    A_G=("A_G", "mean"),
    A_B=("A_B", "mean"),
    photons=("photons", "sum"),
    wl_pixel=("wl_pixel", "mean"),
    n_locs=("frame", "count"),
)
averaged["log_photons"] = np.log10(averaged["photons"])

fig, axs = plotter.one_column_plot(width=1.5, height=2, npanels=2, ratios=[1, 1])

axs[0] = plotter.histogram_plot(axs[0], averaged["wl_pixel"], bins=np.histogram_bin_edges(averaged["wl_pixel"], "fd"),
                                 xlabel="")
mean = np.mean(averaged["wl_pixel"])
axs[0].vlines(x=mean, ymin=0, ymax=0.6, ls="--", color="red", label=r"$\mu$")
std = np.std(averaged["wl_pixel"])
axs[0].axvspan(mean - std, mean + std, alpha=0.15, color="red", label=r"$\sigma$")
axs[0].set_ylabel(r"probability $\rho$")
axs[0].set_xticklabels([])
axs[0].legend(fontsize=6)
axs[1].grid(True, lw=0.5, alpha=0.5, ls="--", color="grey")

sns.kdeplot(data=averaged, x="wl_pixel", y="log_photons", ax=axs[1], fill=True)
axs[1].set_xlabel("mean wavelength/nm")
axs[1].set_ylabel("log(photons)")

plt.savefig(FIG_DIR / "SAureus_Analysis_Wavelength.svg", dpi=600, format="svg")
plt.show()

print(f"Mean wavelength : {mean:.1f} nm")
print(f"Std wavelength  : {std:.1f} nm")


---
## 12 — Fourier ring correlation (FRC) resolution

Uses `PIXEL_SIZE_NM` (71.5 nm, ZWO) — the dev notebook's FRC cell was one of the places
that hardcoded the Ximea value.


In [ ]:
zoom = 8
sr_px_nm = PIXEL_SIZE_NM / zoom

red_xy = linked[["xc", "yc"]].to_numpy()
x_min, y_min = red_xy[:, 0].min(), red_xy[:, 1].min()
red_xy_shifted = red_xy.copy()
red_xy_shifted[:, 0] -= x_min
red_xy_shifted[:, 1] -= y_min

nx_eff = int(red_xy[:, 0].max() - x_min) + 1
ny_eff = int(red_xy[:, 1].max() - y_min) + 1
print(f"Effective field: {nx_eff} x {ny_eff} px ({nx_eff * PIXEL_SIZE_NM:.0f} x {ny_eff * PIXEL_SIZE_NM:.0f} nm)")

resolution_nm, frc_mean, res_hi_nm, res_lo_nm = fire(
    positions=red_xy_shifted, nx=nx_eff, ny=ny_eff, zoom=zoom, n_blocks=50, pixel_size_nm=PIXEL_SIZE_NM,
)
print(f"FIRE resolution: {resolution_nm:.1f} nm")


In [ ]:
fig, ax = plotter.two_column_plot()

sz = max(nx_eff, ny_eff) * zoom
k = np.arange(len(frc_mean))
q = k / sz                    # cycles per SR pixel
q_nm = q / sr_px_nm            # cycles per nm
mask = (q > 0) & (q < 0.6)

q_cross = 1.0 / resolution_nm
q_hi = 1.0 / res_hi_nm
q_lo = 1.0 / res_lo_nm

ax = plotter.line_plot(ax, q_nm[mask], frc_mean[mask], label="FRC (mean), NR4A", color="darkred")
ax.axhline(1 / 7, color="gray", ls="--", lw=1, label="1/7 threshold")
ax.axvline(q_cross, color="crimson", ls="-", lw=1, label=f"FIRE = {resolution_nm:.0f} nm")
ax.axvspan(q_lo, q_hi, alpha=0.15, color="crimson", label=r"$\pm 1\sigma$")

ax.set_xlabel("1 / spatial frequency (nm)", fontsize=8)
ax.set_ylabel("FRC", fontsize=8)
ax.set_ylim([0, 1.05])
ax.set_xscale("log")

tick_nm = [1000, 500, 200, 100, 50, 30, 20]
ax.set_xticks([1 / d for d in tick_nm])
ax.set_xticklabels([f"{d} nm" for d in tick_nm])
ax.set_xlim(1 / 1000, 1 / 15)
ax.legend(fontsize=7)

plt.savefig(FIG_DIR / "FRC_SAureus.svg", dpi=600, format="svg")
plt.show()


---
## Checklist

- [ ] `FAST_MODE` choice confirmed (illustrative vs. full ~230 GB reproduction)
- [ ] Downloaded data matches `biostudies_checksums.json`'s recorded size/md5 for these files
      (spot-check with `md5sum` if in doubt)
- [ ] Raw fit (§3) completes without error
- [ ] Drift trace (§5) looks physically sensible (smooth, bounded — not runaway)
- [ ] Rendered image (§6) shows recognisable bacterial morphology
- [ ] Region A/B rectangles (§10) re-picked against the real rendered image if they land
      on empty background rather than a real spectral boundary
- [ ] FIRE resolution (§12) sanity-checked against the paper's reported value
- [ ] Figure number filled in at the top once assigned
